# encoder-decoder-symmetric composite — cx7: encoder -> bottleneck Linear -> decoder, end-to-end shape parity

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `bottleneck-latent-projection`, `encoder-decoder-symmetric`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "encoder-decoder-symmetric"
DD_ATOM_IDS = ["bottleneck-latent-projection", "encoder-decoder-symmetric"]
DD_SUBTOPICS = ["Generative: Bottleneck latent projection", "CNN: Encoder-decoder symmetric layout"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

An MNIST-style autoencoder threads three blocks together:
1. **Encoder** — convolutional downsampler that takes `(B, 1, 28, 28)` and produces a feature map (e.g. `(B, 32, 7, 7)`). Spatial dims shrink by some power of 2.
2. **Bottleneck** — flatten the spatial map and project DOWN with a single `nn.Linear` to `(B, latent_dim)`. This is the `bottleneck-latent-projection` atom: nothing but a Linear, no activation.
3. **Decoder** — mirror image of the encoder: a Linear to project the latent back UP to the same flattened size, an unflatten/reshape back to `(B, C, H, W)`, then upsampling convs that mirror each encoder downsample.

**Why both atoms together.** The bottleneck is the COMPRESSION; the symmetric layout is the COMMUTATIVE DIAGRAM around it. Without symmetric upsampling you can't reconstruct back to `(B, 1, 28, 28)`. Without the bottleneck you have no compression at all — just a fancy conv net.

**Anatomy.**
```python
# encoder-decoder-symmetric: spatial /= 4 then *= 4.
encoder_conv = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28 -> 14.
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14 -> 7.
)
# bottleneck-latent-projection: flatten + Linear, no activation.
encode_to_latent = nn.Linear(32 * 7 * 7, latent_dim)
decode_from_latent = nn.Linear(latent_dim, 32 * 7 * 7)
decoder_conv = nn.Sequential(
    nn.Upsample(scale_factor=2),
    nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),                    # 7 -> 14.
    nn.Upsample(scale_factor=2),
    nn.Conv2d(16, 1, 3, padding=1),                                # 14 -> 28.
)
```

The test asserts `model(x).shape == x.shape`, that the LATENT shape is `(B, latent_dim)`, and that the bottleneck Linear has NO nonlinearity after it (negative latents must survive the encode->decode path).

### Composite Exercise — encoder -> bottleneck Linear -> decoder, end-to-end shape parity

**Atoms exercised together**: `bottleneck-latent-projection`, `encoder-decoder-symmetric`

Implement `cx7_make_autoencoder(latent_dim)` — return an instance of a tiny MNIST autoencoder with three pieces.

Signature: the returned module accepts `(B, 1, 28, 28)` and returns `(B, 1, 28, 28)`. It must also expose an `.encode(x)` method that returns `(B, latent_dim)`.

Required structure (subclass `nn.Module`):
1. `super().__init__()` first.
2. `self.encoder_conv = nn.Sequential(`
       `nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),`
       `nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),`
   `)`  # (B, 1, 28, 28) -> (B, 32, 7, 7).
3. `self.encode_to_latent = nn.Linear(32 * 7 * 7, latent_dim)`  # bottleneck.
4. `self.decode_from_latent = nn.Linear(latent_dim, 32 * 7 * 7)`
5. `self.decoder_conv = nn.Sequential(`
       `nn.Upsample(scale_factor=2), nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.ReLU(),`
       `nn.Upsample(scale_factor=2), nn.Conv2d(16, 1, kernel_size=3, padding=1),`
   `)`  # (B, 32, 7, 7) -> (B, 1, 28, 28).
6. `encode(self, x)`:
   - run `x` through `encoder_conv` -> `(B, 32, 7, 7)`,
   - flatten to `(B, 32*7*7)`,
   - apply `encode_to_latent` -> `(B, latent_dim)`, RETURN this (no activation).
7. `forward(self, x)`:
   - call `self.encode(x)`,
   - run latent through `decode_from_latent` -> `(B, 32*7*7)`,
   - reshape to `(B, 32, 7, 7)`,
   - run through `decoder_conv` -> `(B, 1, 28, 28)` and return.

No final ReLU/sigmoid on the decoder output — raw linear pixel space.

The test checks: end-to-end shape parity on multiple batch sizes, latent shape `(B, latent_dim)`, that `encode_to_latent` has no activation after it (negative latents pass through), and that all four expected sub-modules exist as named children.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx7_make_autoencoder(latent_dim: int):
    """Return an instance of a tiny conv autoencoder with a Linear bottleneck."""
    raise NotImplementedError

def _test_cx7():
    t.manual_seed(0)
    model = cx7_make_autoencoder(latent_dim=8)
    assert isinstance(model, nn.Module)

    # Case A: required named children all present.
    named = dict(model.named_children())
    for key in ['encoder_conv', 'encode_to_latent', 'decode_from_latent', 'decoder_conv']:
        assert key in named, f'missing child module {key!r}; got {list(named)}'
    assert isinstance(named['encode_to_latent'], nn.Linear)
    assert isinstance(named['decode_from_latent'], nn.Linear)
    assert named['encode_to_latent'].in_features == 32 * 7 * 7, (
        f'bottleneck in_features should be 32*7*7=1568, got {named["encode_to_latent"].in_features}'
    )
    assert named['encode_to_latent'].out_features == 8
    assert named['decode_from_latent'].in_features == 8
    assert named['decode_from_latent'].out_features == 32 * 7 * 7

    # Case B: shape parity end-to-end across multiple batch sizes.
    for B in (1, 3, 8):
        x = t.randn(B, 1, 28, 28)
        out = model(x)
        assert out.shape == x.shape, f'shape parity broken: in={tuple(x.shape)}, out={tuple(out.shape)}'

    # Case C: encode() returns (B, latent_dim).
    x = t.randn(5, 1, 28, 28)
    z = model.encode(x)
    assert z.shape == (5, 8), f'encode() should return (B, latent_dim)=(5, 8); got {tuple(z.shape)}'

    # Case D: bottleneck has NO activation — negative latents must pass through.
    # Construct an input that forces large NEGATIVE pre-activation through the bottleneck.
    with t.no_grad():
        model.encode_to_latent.weight.zero_()
        model.encode_to_latent.bias.fill_(-3.7)
    z2 = model.encode(t.randn(2, 1, 28, 28))
    # Every entry should be ~-3.7 (no ReLU clipping to 0).
    assert t.allclose(z2, -3.7 * t.ones_like(z2), atol=1e-5), (
        f'bottleneck should have NO activation — negatives must survive. Got max {z2.max().item():.3f}, '
        f'min {z2.min().item():.3f}'
    )

    # Case E: latent dim is the BOTTLENECK — smaller than 32*7*7.
    assert 8 < 32 * 7 * 7, 'sanity check: latent dim must be smaller than flattened spatial features'
    _dd_passed.add('cx7')

_test_cx7()

<details><summary>Show solution — cx7</summary>

```python
def cx7_make_autoencoder(latent_dim: int):
    class TinyAE(nn.Module):
        def __init__(self, latent_dim):
            super().__init__()
            # Atom B (encoder-decoder-symmetric): two pool stages -> 28/2/2 = 7.
            self.encoder_conv = nn.Sequential(
                nn.Conv2d(1, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )
            # Atom A (bottleneck-latent-projection): bare Linear, no activation.
            self.encode_to_latent = nn.Linear(32 * 7 * 7, latent_dim)
            self.decode_from_latent = nn.Linear(latent_dim, 32 * 7 * 7)
            # Atom B mirrored: two upsamples mirror the two pools.
            self.decoder_conv = nn.Sequential(
                nn.Upsample(scale_factor=2),
                nn.Conv2d(32, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Upsample(scale_factor=2),
                nn.Conv2d(16, 1, kernel_size=3, padding=1),
            )

        def encode(self, x):
            h = self.encoder_conv(x)            # (B, 32, 7, 7).
            h = h.flatten(start_dim=1)          # (B, 32*7*7).
            return self.encode_to_latent(h)     # (B, latent_dim).

        def forward(self, x):
            z = self.encode(x)
            h = self.decode_from_latent(z)      # (B, 32*7*7).
            h = h.view(h.shape[0], 32, 7, 7)
            return self.decoder_conv(h)

    return TinyAE(latent_dim)
```

The bottleneck is the *only* place where information has to flow through fewer than `32*7*7 = 1568` features — that's what makes the AE actually learn a compression. The `flatten(start_dim=1)` keeps the batch axis intact; `.view(B, 32, 7, 7)` reshapes back before the upsampling stack. Note: no final ReLU/sigmoid on `decoder_conv` — raw linear pixel-space output works for normalized images. For [0, 1] pixels you would add a `nn.Sigmoid()` at the end.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx7',
        'subtopics': ["Generative: Bottleneck latent projection", "CNN: Encoder-decoder symmetric layout"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()